# Production-Grade TransUNet Training Pipeline for Stroke Segmentation

This notebook trains and evaluates the official **Beckschen/TransUNet** 2D architecture on a dataset stored in `stroke_dataset`.

It is organized as a readable, auditable pipeline:

1. Environment and reproducibility
2. Configuration
3. Dataset discovery and validation
4. Data splitting and leakage checks
5. Augmentation and data loaders
6. Official TransUNet model construction
7. Losses and metrics
8. Training and validation
9. Checkpointing and early stopping
10. Test evaluation
11. Qualitative review and artifact export

### Expected dataset layouts

The loader automatically supports either:

```text
stroke_dataset/
├── train/
│   ├── images/
│   └── masks/
├── val/                 # or validation/
│   ├── images/
│   └── masks/
└── test/
    ├── images/
    └── masks/
```

or an unsplit dataset:

```text
stroke_dataset/
├── images/
└── masks/
```

For an unsplit dataset, this notebook creates a deterministic **70/15/15** split.

Images and masks must share the same filename stem, for example:

```text
images/case_001.png
masks/case_001.png
```

Binary masks may contain `{0, 1}` or `{0, 255}`. They are converted to class IDs `{0, 1}`.

## 1. Install dependencies

The official repository was originally documented with an older Python environment. This notebook uses a modern PyTorch workflow while importing the official TransUNet model implementation.

Restart the kernel after installation if your notebook environment requires it.

In [ ]:
# Uncomment and run once in a fresh environment.
# %pip install -q \
#   torch torchvision \
#   numpy pandas matplotlib pillow opencv-python-headless \
#   albumentations scikit-learn scipy tqdm tensorboard ml-collections

# Clone the official implementation when it is not already available.
# !test -d TransUNet || git clone --depth 1 https://github.com/Beckschen/TransUNet.git

## 2. Imports and environment checks

In [ ]:
from __future__ import annotations

import copy
import csv
import hashlib
import json
import math
import os
import random
import shutil
import subprocess
import sys
import time
from dataclasses import asdict, dataclass
from pathlib import Path
from typing import Dict, Iterable, List, Optional, Sequence, Tuple

import albumentations as A
import cv2
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from albumentations.pytorch import ToTensorV2
from PIL import Image
from scipy.ndimage import binary_erosion, distance_transform_edt
from sklearn.model_selection import train_test_split
from torch.cuda.amp import GradScaler, autocast
from torch.utils.data import DataLoader, Dataset
from torch.utils.tensorboard import SummaryWriter
from tqdm.auto import tqdm

print("Python:", sys.version.split()[0])
print("PyTorch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

## 3. Central configuration

In [ ]:
@dataclass(frozen=True)
class Config:
    # Paths
    dataset_root: str = "stroke_dataset"
    transunet_repo: str = "TransUNet"
    output_root: str = "outputs/transunet_stroke"
    pretrained_npz: str = "model/vit_checkpoint/imagenet21k/R50+ViT-B_16.npz"

    # Dataset
    train_ratio: float = 0.70
    val_ratio: float = 0.15
    test_ratio: float = 0.15
    image_size: int = 224
    num_classes: int = 2  # background + stroke
    ignore_index: int = 255
    seed: int = 42

    # Pairing / leakage controls
    image_extensions: Tuple[str, ...] = (".png", ".jpg", ".jpeg", ".tif", ".tiff", ".bmp")
    mask_extensions: Tuple[str, ...] = (".png", ".jpg", ".jpeg", ".tif", ".tiff", ".bmp")
    # Optional: set a delimiter to infer patient ID from a filename.
    # Example "patient001_slice003" with delimiter "_slice" -> "patient001".
    patient_id_delimiter: Optional[str] = None

    # Official TransUNet architecture
    vit_name: str = "R50-ViT-B_16"
    vit_patch_size: int = 16
    n_skip: int = 3
    use_pretrained: bool = True

    # Training
    epochs: int = 100
    batch_size: int = 8
    num_workers: int = 4
    learning_rate: float = 1e-4
    weight_decay: float = 1e-4
    warmup_epochs: int = 5
    early_stopping_patience: int = 15
    gradient_clip_norm: float = 1.0
    amp: bool = True

    # Loss
    ce_weight: float = 0.5
    dice_weight: float = 0.5
    foreground_dice_weight: float = 1.0

    # Runtime
    deterministic: bool = True
    pin_memory: bool = True
    persistent_workers: bool = True

CFG = Config()

assert math.isclose(CFG.train_ratio + CFG.val_ratio + CFG.test_ratio, 1.0)
assert CFG.num_classes >= 2, "This notebook uses multiclass logits: background + foreground class(es)."

DATASET_ROOT = Path(CFG.dataset_root).resolve()
REPO_ROOT = Path(CFG.transunet_repo).resolve()
OUTPUT_ROOT = Path(CFG.output_root).resolve()
CHECKPOINT_DIR = OUTPUT_ROOT / "checkpoints"
LOG_DIR = OUTPUT_ROOT / "logs"
PREDICTION_DIR = OUTPUT_ROOT / "predictions"

for directory in (OUTPUT_ROOT, CHECKPOINT_DIR, LOG_DIR, PREDICTION_DIR):
    directory.mkdir(parents=True, exist_ok=True)

with (OUTPUT_ROOT / "config.json").open("w") as f:
    json.dump(asdict(CFG), f, indent=2)

print(json.dumps(asdict(CFG), indent=2))

## 4. Reproducibility and device setup

In [ ]:
def seed_everything(seed: int, deterministic: bool = True) -> None:
    os.environ["PYTHONHASHSEED"] = str(seed)
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

    if deterministic:
        torch.backends.cudnn.benchmark = False
        torch.backends.cudnn.deterministic = True
        try:
            torch.use_deterministic_algorithms(True, warn_only=True)
        except TypeError:
            torch.use_deterministic_algorithms(True)
    else:
        torch.backends.cudnn.benchmark = True
        torch.backends.cudnn.deterministic = False

seed_everything(CFG.seed, CFG.deterministic)
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", DEVICE)

## 5. Make the official TransUNet repository importable

The notebook imports the official `VisionTransformer` and configuration registry directly from:

```python
networks.vit_seg_modeling
```

In [ ]:
if not REPO_ROOT.exists():
    raise FileNotFoundError(
        f"Official TransUNet repository not found at {REPO_ROOT}.\n"
        "Clone it by running:\n"
        "git clone --depth 1 https://github.com/Beckschen/TransUNet.git"
    )

if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

from networks.vit_seg_modeling import VisionTransformer as ViT_seg
from networks.vit_seg_modeling import CONFIGS as CONFIGS_ViT_seg

print("Official TransUNet import succeeded.")
print("Available configurations:", sorted(CONFIGS_ViT_seg.keys()))

## 6. Dataset discovery and image-mask pairing

In [ ]:
@dataclass(frozen=True)
class Sample:
    sample_id: str
    image_path: str
    mask_path: str
    patient_id: str


def infer_patient_id(stem: str, delimiter: Optional[str]) -> str:
    if delimiter and delimiter in stem:
        return stem.split(delimiter, 1)[0]
    return stem


def indexed_files(folder: Path, extensions: Sequence[str]) -> Dict[str, Path]:
    if not folder.exists():
        raise FileNotFoundError(f"Missing folder: {folder}")

    extensions = {e.lower() for e in extensions}
    files = [p for p in folder.rglob("*") if p.is_file() and p.suffix.lower() in extensions]
    index: Dict[str, Path] = {}

    for path in files:
        stem = path.stem
        if stem in index:
            raise ValueError(
                f"Duplicate filename stem '{stem}' in {folder}:\n"
                f"  {index[stem]}\n  {path}\n"
                "Use unique stems or customize the pairing logic."
            )
        index[stem] = path

    if not index:
        raise RuntimeError(f"No supported files found in {folder}")
    return index


def pair_folder(image_dir: Path, mask_dir: Path) -> List[Sample]:
    images = indexed_files(image_dir, CFG.image_extensions)
    masks = indexed_files(mask_dir, CFG.mask_extensions)

    missing_masks = sorted(set(images) - set(masks))
    missing_images = sorted(set(masks) - set(images))

    if missing_masks or missing_images:
        raise ValueError(
            "Image-mask pairing failed.\n"
            f"Missing masks for {len(missing_masks)} images: {missing_masks[:10]}\n"
            f"Missing images for {len(missing_images)} masks: {missing_images[:10]}"
        )

    samples = []
    for stem in sorted(images):
        samples.append(
            Sample(
                sample_id=stem,
                image_path=str(images[stem]),
                mask_path=str(masks[stem]),
                patient_id=infer_patient_id(stem, CFG.patient_id_delimiter),
            )
        )
    return samples


def locate_split_dir(root: Path, candidates: Sequence[str]) -> Optional[Path]:
    for name in candidates:
        candidate = root / name
        if candidate.exists():
            return candidate
    return None


def discover_dataset(root: Path) -> Dict[str, List[Sample]]:
    train_dir = locate_split_dir(root, ("train", "training"))
    val_dir = locate_split_dir(root, ("val", "validation", "valid"))
    test_dir = locate_split_dir(root, ("test", "testing"))

    if train_dir and val_dir and test_dir:
        return {
            "train": pair_folder(train_dir / "images", train_dir / "masks"),
            "val": pair_folder(val_dir / "images", val_dir / "masks"),
            "test": pair_folder(test_dir / "images", test_dir / "masks"),
        }

    image_dir = root / "images"
    mask_dir = root / "masks"
    if image_dir.exists() and mask_dir.exists():
        return {"all": pair_folder(image_dir, mask_dir)}

    raise FileNotFoundError(
        f"Could not recognize dataset layout under {root}.\n"
        "Expected split folders containing images/ and masks/, "
        "or top-level images/ and masks/."
    )


raw_splits = discover_dataset(DATASET_ROOT)
{k: len(v) for k, v in raw_splits.items()}

## 7. Deterministic 70/15/15 split

When patient IDs are available, splitting is done by patient to prevent slices from the same patient appearing in different partitions. Otherwise each image is treated as one independent case.

In [ ]:
def split_unsplit_samples(samples: List[Sample]) -> Dict[str, List[Sample]]:
    groups: Dict[str, List[Sample]] = {}
    for sample in samples:
        groups.setdefault(sample.patient_id, []).append(sample)

    group_ids = sorted(groups)
    if len(group_ids) < 3:
        raise ValueError("At least three independent patients/cases are required for train/val/test splits.")

    train_groups, temp_groups = train_test_split(
        group_ids,
        test_size=CFG.val_ratio + CFG.test_ratio,
        random_state=CFG.seed,
        shuffle=True,
    )

    relative_test_ratio = CFG.test_ratio / (CFG.val_ratio + CFG.test_ratio)
    val_groups, test_groups = train_test_split(
        temp_groups,
        test_size=relative_test_ratio,
        random_state=CFG.seed,
        shuffle=True,
    )

    def expand(ids: Sequence[str]) -> List[Sample]:
        return sorted(
            [sample for group_id in ids for sample in groups[group_id]],
            key=lambda x: x.sample_id,
        )

    return {
        "train": expand(train_groups),
        "val": expand(val_groups),
        "test": expand(test_groups),
    }


splits = (
    split_unsplit_samples(raw_splits["all"])
    if "all" in raw_splits
    else raw_splits
)

split_summary = pd.DataFrame(
    [
        {
            "split": name,
            "images": len(samples),
            "patients_or_cases": len({s.patient_id for s in samples}),
            "ratio": len(samples) / sum(len(v) for v in splits.values()),
        }
        for name, samples in splits.items()
    ]
)
split_summary

## 8. Leakage, duplicates, corruption, and mask-value checks

In [ ]:
def sha256_file(path: str, chunk_size: int = 1024 * 1024) -> str:
    digest = hashlib.sha256()
    with open(path, "rb") as f:
        for chunk in iter(lambda: f.read(chunk_size), b""):
            digest.update(chunk)
    return digest.hexdigest()


def read_image(path: str) -> np.ndarray:
    image = cv2.imread(path, cv2.IMREAD_UNCHANGED)
    if image is None:
        raise ValueError(f"Unreadable image: {path}")
    if image.ndim == 3:
        image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)
    return image


def read_mask(path: str) -> np.ndarray:
    mask = cv2.imread(path, cv2.IMREAD_UNCHANGED)
    if mask is None:
        raise ValueError(f"Unreadable mask: {path}")
    if mask.ndim == 3:
        # Labelbox exports may be RGB; identical channels are accepted.
        if not (np.array_equal(mask[..., 0], mask[..., 1]) and np.array_equal(mask[..., 1], mask[..., 2])):
            raise ValueError(
                f"RGB mask has non-identical channels: {path}. "
                "Convert color-coded masks to integer class IDs first."
            )
        mask = mask[..., 0]
    return mask


def normalize_binary_mask(mask: np.ndarray) -> np.ndarray:
    values = set(np.unique(mask).tolist())
    if values.issubset({0, 1}):
        return mask.astype(np.uint8)
    if values.issubset({0, 255}):
        return (mask > 0).astype(np.uint8)
    raise ValueError(f"Unexpected binary mask values: {sorted(values)[:20]}")


def validate_dataset(splits: Dict[str, List[Sample]]) -> pd.DataFrame:
    # Patient leakage
    patient_sets = {k: {s.patient_id for s in v} for k, v in splits.items()}
    for left, right in (("train", "val"), ("train", "test"), ("val", "test")):
        overlap = patient_sets[left] & patient_sets[right]
        if overlap:
            raise ValueError(f"Patient/case leakage between {left} and {right}: {sorted(overlap)[:10]}")

    # File-level duplicate leakage by image hash
    hash_to_split: Dict[str, str] = {}
    records = []
    for split_name, samples in splits.items():
        foreground_pixels = 0
        total_pixels = 0
        empty_masks = 0

        for sample in tqdm(samples, desc=f"Validating {split_name}"):
            image = read_image(sample.image_path)
            mask = normalize_binary_mask(read_mask(sample.mask_path))

            if image.shape[:2] != mask.shape[:2]:
                raise ValueError(
                    f"Shape mismatch for {sample.sample_id}: "
                    f"image={image.shape}, mask={mask.shape}"
                )

            image_hash = sha256_file(sample.image_path)
            previous_split = hash_to_split.get(image_hash)
            if previous_split and previous_split != split_name:
                raise ValueError(
                    f"Duplicate image content found across {previous_split} and {split_name}: "
                    f"{sample.image_path}"
                )
            hash_to_split[image_hash] = split_name

            fg = int(mask.sum())
            foreground_pixels += fg
            total_pixels += mask.size
            empty_masks += int(fg == 0)

        records.append(
            {
                "split": split_name,
                "samples": len(samples),
                "empty_masks": empty_masks,
                "empty_mask_pct": 100 * empty_masks / len(samples),
                "foreground_pixel_pct": 100 * foreground_pixels / total_pixels,
            }
        )

    return pd.DataFrame(records)


quality_report = validate_dataset(splits)
quality_report

## 9. Persist the exact split manifest

In [ ]:
manifest_rows = []
for split_name, samples in splits.items():
    for sample in samples:
        manifest_rows.append(
            {
                "split": split_name,
                "sample_id": sample.sample_id,
                "patient_id": sample.patient_id,
                "image_path": sample.image_path,
                "mask_path": sample.mask_path,
            }
        )

manifest_df = pd.DataFrame(manifest_rows)
manifest_path = OUTPUT_ROOT / "split_manifest.csv"
manifest_df.to_csv(manifest_path, index=False)
print("Saved:", manifest_path)
manifest_df.head()

## 10. Visual dataset sanity check

In [ ]:
def show_samples(samples: Sequence[Sample], n: int = 4) -> None:
    chosen = random.sample(list(samples), k=min(n, len(samples)))
    fig, axes = plt.subplots(len(chosen), 3, figsize=(12, 4 * len(chosen)))
    axes = np.atleast_2d(axes)

    for row, sample in enumerate(chosen):
        image = read_image(sample.image_path)
        mask = normalize_binary_mask(read_mask(sample.mask_path))

        if image.ndim == 2:
            display_image = image
            cmap = "gray"
        else:
            display_image = image
            cmap = None

        axes[row, 0].imshow(display_image, cmap=cmap)
        axes[row, 0].set_title(f"{sample.sample_id}: image")
        axes[row, 1].imshow(mask, cmap="gray", vmin=0, vmax=1)
        axes[row, 1].set_title("Ground-truth mask")
        axes[row, 2].imshow(display_image, cmap=cmap)
        axes[row, 2].imshow(mask, alpha=0.35, cmap="Reds", vmin=0, vmax=1)
        axes[row, 2].set_title("Overlay")

        for ax in axes[row]:
            ax.axis("off")

    plt.tight_layout()
    plt.show()


show_samples(splits["train"])

## 11. Augmentation pipelines

In [ ]:
train_transform = A.Compose(
    [
        A.Resize(CFG.image_size, CFG.image_size, interpolation=cv2.INTER_LINEAR),
        A.HorizontalFlip(p=0.5),
        A.ShiftScaleRotate(
            shift_limit=0.05,
            scale_limit=0.10,
            rotate_limit=15,
            border_mode=cv2.BORDER_CONSTANT,
            value=0,
            mask_value=0,
            p=0.6,
        ),
        A.RandomBrightnessContrast(
            brightness_limit=0.15,
            contrast_limit=0.15,
            p=0.35,
        ),
        A.GaussNoise(p=0.15),
        A.Normalize(mean=(0.5, 0.5, 0.5), std=(0.5, 0.5, 0.5)),
        ToTensorV2(),
    ]
)

eval_transform = A.Compose(
    [
        A.Resize(CFG.image_size, CFG.image_size, interpolation=cv2.INTER_LINEAR),
        A.Normalize(mean=(0.5, 0.5, 0.5), std=(0.5, 0.5, 0.5)),
        ToTensorV2(),
    ]
)

## 12. PyTorch dataset

In [ ]:
class StrokeSegmentationDataset(Dataset):
    def __init__(self, samples: Sequence[Sample], transform: A.Compose):
        self.samples = list(samples)
        self.transform = transform

    def __len__(self) -> int:
        return len(self.samples)

    def __getitem__(self, index: int) -> Dict[str, object]:
        sample = self.samples[index]
        image = read_image(sample.image_path)
        mask = normalize_binary_mask(read_mask(sample.mask_path))

        # Convert grayscale or single-channel input to RGB for the official model.
        if image.ndim == 2:
            image = np.repeat(image[..., None], 3, axis=2)
        elif image.shape[2] == 1:
            image = np.repeat(image, 3, axis=2)
        elif image.shape[2] > 3:
            image = image[..., :3]

        transformed = self.transform(image=image, mask=mask)
        image_tensor = transformed["image"].float()
        mask_tensor = transformed["mask"].long()

        if mask_tensor.ndim == 3:
            mask_tensor = mask_tensor.squeeze(0)

        return {
            "image": image_tensor,
            "mask": mask_tensor,
            "sample_id": sample.sample_id,
            "original_size": torch.tensor(mask.shape, dtype=torch.int64),
        }


train_dataset = StrokeSegmentationDataset(splits["train"], train_transform)
val_dataset = StrokeSegmentationDataset(splits["val"], eval_transform)
test_dataset = StrokeSegmentationDataset(splits["test"], eval_transform)

batch = train_dataset[0]
print("Image:", batch["image"].shape, batch["image"].dtype)
print("Mask:", batch["mask"].shape, batch["mask"].dtype, torch.unique(batch["mask"]))

## 13. Data loaders

In [ ]:
def seed_worker(worker_id: int) -> None:
    worker_seed = CFG.seed + worker_id
    np.random.seed(worker_seed)
    random.seed(worker_seed)


generator = torch.Generator()
generator.manual_seed(CFG.seed)

loader_kwargs = dict(
    batch_size=CFG.batch_size,
    num_workers=CFG.num_workers,
    pin_memory=CFG.pin_memory and torch.cuda.is_available(),
    worker_init_fn=seed_worker,
    generator=generator,
    persistent_workers=CFG.persistent_workers and CFG.num_workers > 0,
)

train_loader = DataLoader(train_dataset, shuffle=True, drop_last=False, **loader_kwargs)
val_loader = DataLoader(val_dataset, shuffle=False, drop_last=False, **loader_kwargs)
test_loader = DataLoader(test_dataset, shuffle=False, drop_last=False, **loader_kwargs)

print("Batches:", len(train_loader), len(val_loader), len(test_loader))

## 14. Build the official TransUNet model

For `R50-ViT-B_16`, the official training script adjusts the hybrid patch grid to:

```python
(image_size / vit_patch_size, image_size / vit_patch_size)
```

and loads ImageNet-21k `.npz` weights when available.

In [ ]:
def build_transunet() -> nn.Module:
    if CFG.vit_name not in CONFIGS_ViT_seg:
        raise KeyError(
            f"Unknown ViT configuration '{CFG.vit_name}'. "
            f"Available: {sorted(CONFIGS_ViT_seg)}"
        )

    config_vit = copy.deepcopy(CONFIGS_ViT_seg[CFG.vit_name])
    config_vit.n_classes = CFG.num_classes
    config_vit.n_skip = CFG.n_skip

    if "R50" in CFG.vit_name:
        grid_size = int(CFG.image_size / CFG.vit_patch_size)
        config_vit.patches.grid = (grid_size, grid_size)

    model = ViT_seg(
        config_vit,
        img_size=CFG.image_size,
        num_classes=CFG.num_classes,
    )

    pretrained_path = REPO_ROOT / CFG.pretrained_npz
    if CFG.use_pretrained:
        if not pretrained_path.exists():
            raise FileNotFoundError(
                f"Pretrained weights requested but not found at:\n{pretrained_path}\n\n"
                "Place R50+ViT-B_16.npz there or set use_pretrained=False."
            )
        weights = np.load(pretrained_path)
        model.load_from(weights=weights)
        print("Loaded pretrained weights:", pretrained_path)
    else:
        print("Training without ImageNet-21k initialization.")

    return model


model = build_transunet().to(DEVICE)

trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
total_params = sum(p.numel() for p in model.parameters())
print(f"Parameters: {total_params:,} total | {trainable_params:,} trainable")

## 15. Verify the model output contract

In [ ]:
with torch.no_grad():
    dummy = torch.randn(1, 3, CFG.image_size, CFG.image_size, device=DEVICE)
    output = model(dummy)

expected = (1, CFG.num_classes, CFG.image_size, CFG.image_size)
assert tuple(output.shape) == expected, f"Expected {expected}, got {tuple(output.shape)}"
print("Output shape:", tuple(output.shape))

## 16. Loss functions

In [ ]:
class SoftDiceLoss(nn.Module):
    def __init__(self, num_classes: int, smooth: float = 1e-5, include_background: bool = False):
        super().__init__()
        self.num_classes = num_classes
        self.smooth = smooth
        self.include_background = include_background

    def forward(self, logits: torch.Tensor, targets: torch.Tensor) -> torch.Tensor:
        probabilities = torch.softmax(logits, dim=1)
        one_hot = F.one_hot(
            targets.clamp(min=0, max=self.num_classes - 1),
            num_classes=self.num_classes,
        ).permute(0, 3, 1, 2).float()

        start_class = 0 if self.include_background else 1
        probabilities = probabilities[:, start_class:]
        one_hot = one_hot[:, start_class:]

        dims = (0, 2, 3)
        intersection = torch.sum(probabilities * one_hot, dims)
        denominator = torch.sum(probabilities + one_hot, dims)
        dice = (2 * intersection + self.smooth) / (denominator + self.smooth)
        return 1 - dice.mean()


cross_entropy = nn.CrossEntropyLoss(ignore_index=CFG.ignore_index)
dice_loss_fn = SoftDiceLoss(CFG.num_classes, include_background=False)


def segmentation_loss(logits: torch.Tensor, targets: torch.Tensor) -> Tuple[torch.Tensor, Dict[str, float]]:
    ce = cross_entropy(logits, targets)
    dice = dice_loss_fn(logits, targets)
    total = CFG.ce_weight * ce + CFG.dice_weight * dice
    return total, {"ce_loss": ce.item(), "dice_loss": dice.item()}

## 17. Segmentation metrics

In [ ]:
def confusion_counts(
    predictions: torch.Tensor,
    targets: torch.Tensor,
    positive_class: int = 1,
) -> Tuple[int, int, int, int]:
    pred_pos = predictions == positive_class
    true_pos = targets == positive_class

    tp = int(torch.sum(pred_pos & true_pos).item())
    fp = int(torch.sum(pred_pos & ~true_pos).item())
    fn = int(torch.sum(~pred_pos & true_pos).item())
    tn = int(torch.sum(~pred_pos & ~true_pos).item())
    return tp, fp, fn, tn


def metrics_from_counts(tp: int, fp: int, fn: int, tn: int, eps: float = 1e-7) -> Dict[str, float]:
    return {
        "dice": (2 * tp + eps) / (2 * tp + fp + fn + eps),
        "iou": (tp + eps) / (tp + fp + fn + eps),
        "precision": (tp + eps) / (tp + fp + eps),
        "recall": (tp + eps) / (tp + fn + eps),
        "specificity": (tn + eps) / (tn + fp + eps),
        "accuracy": (tp + tn + eps) / (tp + fp + fn + tn + eps),
    }


def hd95_binary(pred: np.ndarray, target: np.ndarray) -> float:
    pred = pred.astype(bool)
    target = target.astype(bool)

    if not pred.any() and not target.any():
        return 0.0
    if not pred.any() or not target.any():
        return float("nan")

    pred_border = pred ^ binary_erosion(pred)
    target_border = target ^ binary_erosion(target)

    target_distance = distance_transform_edt(~target_border)
    pred_distance = distance_transform_edt(~pred_border)

    distances = np.concatenate(
        [target_distance[pred_border], pred_distance[target_border]]
    )
    return float(np.percentile(distances, 95))

## 18. Optimizer, warmup + cosine scheduler, and AMP

In [ ]:
optimizer = torch.optim.AdamW(
    model.parameters(),
    lr=CFG.learning_rate,
    weight_decay=CFG.weight_decay,
)

def lr_lambda(epoch: int) -> float:
    if epoch < CFG.warmup_epochs:
        return float(epoch + 1) / max(1, CFG.warmup_epochs)

    progress = (epoch - CFG.warmup_epochs) / max(1, CFG.epochs - CFG.warmup_epochs)
    return 0.5 * (1.0 + math.cos(math.pi * progress))

scheduler = torch.optim.lr_scheduler.LambdaLR(optimizer, lr_lambda=lr_lambda)
scaler = GradScaler(enabled=CFG.amp and torch.cuda.is_available())

## 19. Training and validation epoch functions

In [ ]:
def run_epoch(
    model: nn.Module,
    loader: DataLoader,
    training: bool,
) -> Dict[str, float]:
    model.train(training)

    total_loss = 0.0
    total_ce = 0.0
    total_dice_loss = 0.0
    total_samples = 0
    tp = fp = fn = tn = 0

    progress = tqdm(loader, desc="train" if training else "validate", leave=False)

    for batch in progress:
        images = batch["image"].to(DEVICE, non_blocking=True)
        targets = batch["mask"].to(DEVICE, non_blocking=True)
        batch_size = images.size(0)

        if training:
            optimizer.zero_grad(set_to_none=True)

        with torch.set_grad_enabled(training):
            with autocast(enabled=scaler.is_enabled()):
                logits = model(images)
                loss, parts = segmentation_loss(logits, targets)

            if training:
                scaler.scale(loss).backward()
                scaler.unscale_(optimizer)
                torch.nn.utils.clip_grad_norm_(model.parameters(), CFG.gradient_clip_norm)
                scaler.step(optimizer)
                scaler.update()

        predictions = torch.argmax(logits.detach(), dim=1)
        batch_counts = confusion_counts(predictions, targets)
        tp += batch_counts[0]
        fp += batch_counts[1]
        fn += batch_counts[2]
        tn += batch_counts[3]

        total_loss += loss.item() * batch_size
        total_ce += parts["ce_loss"] * batch_size
        total_dice_loss += parts["dice_loss"] * batch_size
        total_samples += batch_size

        progress.set_postfix(loss=f"{loss.item():.4f}")

    metrics = metrics_from_counts(tp, fp, fn, tn)
    metrics.update(
        {
            "loss": total_loss / total_samples,
            "ce_loss": total_ce / total_samples,
            "dice_loss": total_dice_loss / total_samples,
        }
    )
    return metrics

## 20. Atomic checkpoint helpers

In [ ]:
def save_checkpoint_atomic(payload: Dict[str, object], path: Path) -> None:
    temporary = path.with_suffix(path.suffix + ".tmp")
    torch.save(payload, temporary)
    os.replace(temporary, path)


def checkpoint_payload(epoch: int, best_val_dice: float, history: List[Dict[str, float]]) -> Dict[str, object]:
    return {
        "epoch": epoch,
        "model_state_dict": model.state_dict(),
        "optimizer_state_dict": optimizer.state_dict(),
        "scheduler_state_dict": scheduler.state_dict(),
        "scaler_state_dict": scaler.state_dict(),
        "best_val_dice": best_val_dice,
        "config": asdict(CFG),
        "history": history,
        "torch_version": torch.__version__,
    }


def load_checkpoint(path: Path, load_optimizer: bool = False) -> Dict[str, object]:
    checkpoint = torch.load(path, map_location=DEVICE)
    model.load_state_dict(checkpoint["model_state_dict"])

    if load_optimizer:
        optimizer.load_state_dict(checkpoint["optimizer_state_dict"])
        scheduler.load_state_dict(checkpoint["scheduler_state_dict"])
        scaler.load_state_dict(checkpoint["scaler_state_dict"])

    return checkpoint

## 21. Train with early stopping, TensorBoard, and CSV logging

In [ ]:
writer = SummaryWriter(log_dir=str(LOG_DIR / "tensorboard"))
history: List[Dict[str, float]] = []
best_val_dice = -float("inf")
epochs_without_improvement = 0

best_path = CHECKPOINT_DIR / "best_transunet.pt"
last_path = CHECKPOINT_DIR / "last_transunet.pt"
history_path = LOG_DIR / "history.csv"

for epoch in range(CFG.epochs):
    started = time.time()

    train_metrics = run_epoch(model, train_loader, training=True)
    val_metrics = run_epoch(model, val_loader, training=False)
    scheduler.step()

    row: Dict[str, float] = {
        "epoch": epoch + 1,
        "learning_rate": optimizer.param_groups[0]["lr"],
        "elapsed_seconds": time.time() - started,
    }
    row.update({f"train_{k}": v for k, v in train_metrics.items()})
    row.update({f"val_{k}": v for k, v in val_metrics.items()})
    history.append(row)

    for key, value in row.items():
        if key != "epoch":
            writer.add_scalar(key, value, epoch + 1)

    pd.DataFrame(history).to_csv(history_path, index=False)
    save_checkpoint_atomic(
        checkpoint_payload(epoch + 1, best_val_dice, history),
        last_path,
    )

    improved = val_metrics["dice"] > best_val_dice
    if improved:
        best_val_dice = val_metrics["dice"]
        epochs_without_improvement = 0
        save_checkpoint_atomic(
            checkpoint_payload(epoch + 1, best_val_dice, history),
            best_path,
        )
    else:
        epochs_without_improvement += 1

    print(
        f"Epoch {epoch + 1:03d}/{CFG.epochs} | "
        f"train loss {train_metrics['loss']:.4f} dice {train_metrics['dice']:.4f} | "
        f"val loss {val_metrics['loss']:.4f} dice {val_metrics['dice']:.4f} | "
        f"best {best_val_dice:.4f}"
    )

    if epochs_without_improvement >= CFG.early_stopping_patience:
        print(f"Early stopping after {CFG.early_stopping_patience} epochs without improvement.")
        break

writer.close()
print("Best checkpoint:", best_path)

## 22. Plot learning curves

In [ ]:
history_df = pd.read_csv(history_path)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
axes[0].plot(history_df["epoch"], history_df["train_loss"], label="train")
axes[0].plot(history_df["epoch"], history_df["val_loss"], label="validation")
axes[0].set_title("Loss")
axes[0].set_xlabel("Epoch")
axes[0].legend()

axes[1].plot(history_df["epoch"], history_df["train_dice"], label="train")
axes[1].plot(history_df["epoch"], history_df["val_dice"], label="validation")
axes[1].set_title("Dice")
axes[1].set_xlabel("Epoch")
axes[1].legend()

plt.tight_layout()
plt.show()

## 23. Load the best model and evaluate the untouched test set

In [ ]:
best_checkpoint = load_checkpoint(best_path)
print(
    "Loaded epoch:",
    best_checkpoint["epoch"],
    "| best validation Dice:",
    best_checkpoint["best_val_dice"],
)
model.eval()

In [ ]:
@torch.no_grad()
def evaluate_test_set(
    model: nn.Module,
    loader: DataLoader,
    save_predictions: bool = True,
) -> Tuple[pd.DataFrame, Dict[str, float]]:
    rows = []

    for batch in tqdm(loader, desc="test"):
        images = batch["image"].to(DEVICE, non_blocking=True)
        targets = batch["mask"].to(DEVICE, non_blocking=True)
        sample_ids = batch["sample_id"]

        with autocast(enabled=scaler.is_enabled()):
            logits = model(images)
        predictions = torch.argmax(logits, dim=1)

        for i, sample_id in enumerate(sample_ids):
            pred = predictions[i].cpu().numpy().astype(np.uint8)
            target = targets[i].cpu().numpy().astype(np.uint8)

            tp_i, fp_i, fn_i, tn_i = confusion_counts(
                predictions[i].cpu(),
                targets[i].cpu(),
            )
            metrics = metrics_from_counts(tp_i, fp_i, fn_i, tn_i)
            metrics["hd95"] = hd95_binary(pred == 1, target == 1)
            metrics["sample_id"] = sample_id
            rows.append(metrics)

            if save_predictions:
                Image.fromarray((pred * 255).astype(np.uint8)).save(
                    PREDICTION_DIR / f"{sample_id}_prediction.png"
                )

    per_case = pd.DataFrame(rows)
    aggregate = {}

    for metric in ("dice", "iou", "precision", "recall", "specificity", "accuracy", "hd95"):
        values = per_case[metric].dropna()
        aggregate[f"{metric}_mean"] = float(values.mean())
        aggregate[f"{metric}_std"] = float(values.std(ddof=1)) if len(values) > 1 else 0.0

    return per_case, aggregate


test_per_case, test_summary = evaluate_test_set(model, test_loader)
test_per_case.to_csv(OUTPUT_ROOT / "test_per_case_metrics.csv", index=False)

with (OUTPUT_ROOT / "test_summary.json").open("w") as f:
    json.dump(test_summary, f, indent=2)

pd.DataFrame([test_summary]).T.rename(columns={0: "value"})

## 24. Bootstrap 95% confidence intervals

In [ ]:
def bootstrap_ci(
    values: Sequence[float],
    statistic=np.mean,
    n_bootstrap: int = 5000,
    confidence: float = 0.95,
    seed: int = 42,
) -> Tuple[float, float]:
    clean = np.asarray([v for v in values if np.isfinite(v)], dtype=float)
    if len(clean) == 0:
        return float("nan"), float("nan")

    rng = np.random.default_rng(seed)
    estimates = np.empty(n_bootstrap, dtype=float)
    for i in range(n_bootstrap):
        sample = rng.choice(clean, size=len(clean), replace=True)
        estimates[i] = statistic(sample)

    alpha = (1 - confidence) / 2
    return (
        float(np.quantile(estimates, alpha)),
        float(np.quantile(estimates, 1 - alpha)),
    )


ci_rows = []
for metric in ("dice", "iou", "precision", "recall", "specificity", "hd95"):
    lower, upper = bootstrap_ci(test_per_case[metric].dropna().values, seed=CFG.seed)
    ci_rows.append(
        {
            "metric": metric,
            "mean": test_per_case[metric].mean(),
            "95% CI lower": lower,
            "95% CI upper": upper,
        }
    )

ci_df = pd.DataFrame(ci_rows)
ci_df.to_csv(OUTPUT_ROOT / "test_confidence_intervals.csv", index=False)
ci_df

## 25. Qualitative test predictions

In [ ]:
@torch.no_grad()
def visualize_predictions(model: nn.Module, dataset: Dataset, n: int = 6) -> None:
    indices = np.linspace(0, len(dataset) - 1, num=min(n, len(dataset)), dtype=int)
    fig, axes = plt.subplots(len(indices), 4, figsize=(15, 4 * len(indices)))
    axes = np.atleast_2d(axes)

    for row, index in enumerate(indices):
        item = dataset[index]
        image_tensor = item["image"].unsqueeze(0).to(DEVICE)

        logits = model(image_tensor)
        pred = torch.argmax(logits, dim=1)[0].cpu().numpy()
        target = item["mask"].numpy()

        image = item["image"].permute(1, 2, 0).numpy()
        image = np.clip(image * 0.5 + 0.5, 0, 1)

        axes[row, 0].imshow(image)
        axes[row, 0].set_title(item["sample_id"])
        axes[row, 1].imshow(target, cmap="gray", vmin=0, vmax=1)
        axes[row, 1].set_title("Ground truth")
        axes[row, 2].imshow(pred, cmap="gray", vmin=0, vmax=1)
        axes[row, 2].set_title("Prediction")
        axes[row, 3].imshow(image)
        axes[row, 3].imshow(pred, alpha=0.35, cmap="Reds", vmin=0, vmax=1)
        axes[row, 3].set_title("Prediction overlay")

        for ax in axes[row]:
            ax.axis("off")

    plt.tight_layout()
    plt.show()


visualize_predictions(model, test_dataset)

## 26. Export a deployment checkpoint

In [ ]:
deployment_payload = {
    "model_state_dict": model.state_dict(),
    "architecture": {
        "name": "TransUNet",
        "vit_name": CFG.vit_name,
        "image_size": CFG.image_size,
        "vit_patch_size": CFG.vit_patch_size,
        "n_skip": CFG.n_skip,
        "num_classes": CFG.num_classes,
    },
    "preprocessing": {
        "resize": [CFG.image_size, CFG.image_size],
        "input_channels": 3,
        "normalization_mean": [0.5, 0.5, 0.5],
        "normalization_std": [0.5, 0.5, 0.5],
        "mask_class_mapping": {"background": 0, "stroke": 1},
    },
    "validation_best_dice": best_checkpoint["best_val_dice"],
    "test_metrics": test_summary,
}

deployment_path = OUTPUT_ROOT / "transunet_stroke_deployment.pt"
save_checkpoint_atomic(deployment_payload, deployment_path)
print("Saved:", deployment_path)

## 27. Operational checklist

Before reporting this experiment:

- Confirm the split manifest is patient-level where multiple slices belong to one patient.
- Record the exact Git commit used from the official TransUNet repository.
- Keep the test set untouched until model and hyperparameters are finalized.
- Review empty-mask cases separately.
- Inspect false positives and false negatives clinically.
- Report Dice, IoU, precision, recall, specificity, HD95, confidence intervals, parameter count, and inference latency.
- Use the same split manifest, preprocessing, metrics, and reporting code for Models 2 and 3.

In [ ]:
def get_git_commit(repo: Path) -> Optional[str]:
    try:
        return subprocess.check_output(
            ["git", "-C", str(repo), "rev-parse", "HEAD"],
            text=True,
        ).strip()
    except Exception:
        return None

run_metadata = {
    "transunet_git_commit": get_git_commit(REPO_ROOT),
    "device": str(DEVICE),
    "gpu": torch.cuda.get_device_name(0) if torch.cuda.is_available() else None,
    "torch_version": torch.__version__,
    "cuda_version": torch.version.cuda,
    "config": asdict(CFG),
}

with (OUTPUT_ROOT / "run_metadata.json").open("w") as f:
    json.dump(run_metadata, f, indent=2)

run_metadata